# 03 · Ajustes metodológicos de comparabilidad

Se añaden las cuatro capas que faltaban para que
la comparación entre administraciones sea **estadísticamente justa y no sesgada**:

1. **Deflactor IPC (pesos constantes).** Los valores de SECOP son nominales. Comparar
   pesos de 2022 con pesos de 2025 sin corregir la inflación **infla artificialmente** a la
   administración más reciente. Se construye un índice IPC mensual (DANE) y valores reales.
2. **Año y mes de gobierno.** Para no comparar el *año 3-4* de un alcalde contra el *año 1-2*
   del otro, se etiqueta cada contrato por su posición dentro del mandato.
3. **Calendario electoral y ley de garantías.** Se marcan la ventana de restricción de
   contratación directa (4 meses antes de elección) y los años electoral/preelectoral,
   componente central del proyecto que aún no estaba operacionalizado.
4. **Ventanas comparables alineadas por mandato.** Se prioriza comparar *los mismos meses de
   gobierno* de cada alcalde, respetando el límite real de cobertura de SECOP.

In [1]:
# 1. Librerías
import json
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.width", 200)

In [2]:
# 2. Detectar rutas del proyecto (misma lógica de los cuadernos previos)
RUTA_ACTUAL = Path.cwd().resolve()
if (RUTA_ACTUAL / "datos").exists() and (RUTA_ACTUAL / "notebooks").exists():
    RUTA_PROYECTO = RUTA_ACTUAL
elif RUTA_ACTUAL.name.lower() == "notebooks":
    RUTA_PROYECTO = RUTA_ACTUAL.parent
elif (RUTA_ACTUAL.parent / "datos").exists():
    RUTA_PROYECTO = RUTA_ACTUAL.parent
else:
    RUTA_PROYECTO = RUTA_ACTUAL

RUTA_INTERMEDIOS = RUTA_PROYECTO / "datos" / "intermedios"
RUTA_PROCESADOS = RUTA_PROYECTO / "datos" / "procesados"
RUTA_TABLAS = RUTA_PROYECTO / "entregables" / "tablas"
RUTA_TABLAS.mkdir(parents=True, exist_ok=True)
print("Raíz del proyecto:", RUTA_PROYECTO)

Raíz del proyecto: D:\Users\LAURA PEREZ\Desktop\CIENCIA DE DATOS\PROYECTO SECOP_BARRANCABERMEJA


In [3]:
# 3. Cargar las bases producidas por el cuaderno 02
base = pd.read_parquet(RUTA_INTERMEDIOS / "02_base_maestra_clasificada.parquet")
print(f"Base maestra clasificada: {len(base):,} contratos, {len(base.columns)} columnas")

# Universos que reconstruiremos con los ajustes ya aplicados
for c in ["fecha_asignacion_periodo"]:
    base[c] = pd.to_datetime(base[c], errors="coerce")

Base maestra clasificada: 37,574 contratos, 123 columnas


## Capa 1 · Deflactor IPC de pesos nominales a pesos constantes

SECOP registra **valores nominales** (pesos del año en que se firmó el contrato). La inflación
en Colombia fue muy alta en el periodo analizado, así que un mismo salario real aparece como
cifras cada vez más grandes con el paso de los años.

Se usa la **variación anual del IPC (diciembre–diciembre) publicada por el DANE** como anclas, y
se interpola un **índice mensual** por crecimiento geométrico entre diciembres. Con ese índice se
llevan todos los valores a **pesos constantes de mediados de 2025** (año base configurable).

> **Fuente:** DANE, IPC base 2018. La variación de diciembre de 2025 fue **5,10 %**.
> Para la versión de publicación se recomienda reemplazar el índice interpolado por la
> **serie mensual oficial del IPC** (mismo hook de código, misma columna `ipc_indice`).


In [4]:
# 4. Parámetros del deflactor IPC (editables). Variación anual DANE, diciembre-diciembre.
INFLACION_DIC = {
    2020: 0.0161,   # DANE
    2021: 0.0562,
    2022: 0.1312,
    2023: 0.0928,
    2024: 0.0520,
    2025: 0.0510,   # dic-2025 confirmado por DANE
}
INFLACION_2026_PROYECTADA = 0.048   # proyección editable para el año en curso
ANIO_BASE = 2025                     # pesos constantes de mediados de este año
MES_BASE = 6

# Índice de diciembre (base dic-2019 = 100)
indice_dic = {2019: 100.0}
for anio in range(2020, 2026):
    indice_dic[anio] = indice_dic[anio - 1] * (1 + INFLACION_DIC[anio])
indice_dic[2026] = indice_dic[2025] * (1 + INFLACION_2026_PROYECTADA)

def indice_ipc_mensual(anio, mes):
    """Índice IPC del mes por interpolación geométrica entre diciembres consecutivos."""
    if pd.isna(anio) or pd.isna(mes):
        return np.nan
    anio, mes = int(anio), int(mes)
    if (anio - 1) not in indice_dic or anio not in indice_dic:
        return np.nan
    ini, fin = indice_dic[anio - 1], indice_dic[anio]
    return ini * (fin / ini) ** (mes / 12.0)

INDICE_BASE = indice_ipc_mensual(ANIO_BASE, MES_BASE)
print("Índice IPC diciembre por año:")
for a, v in indice_dic.items():
    print(f"  {a}: {v:6.2f}")
print(f"\nÍndice base (mediados {ANIO_BASE}): {INDICE_BASE:.2f}")

Índice IPC diciembre por año:
  2019: 100.00
  2020: 101.61
  2021: 107.32
  2022: 121.40
  2023: 132.67
  2024: 139.57
  2025: 146.68
  2026: 153.72

Índice base (mediados 2025): 143.08


In [5]:
# 5. Aplicar el deflactor a toda la base
base["ipc_indice"] = [
    indice_ipc_mensual(a, m) for a, m in zip(base["anio_periodo"], base["mes_periodo"])
]
base["deflactor_ipc"] = INDICE_BASE / base["ipc_indice"]

base["valor_contrato_real"] = base["valor_contrato_num"] * base["deflactor_ipc"]
base["valor_mensual_real"] = base["valor_mensual_equivalente"] * base["deflactor_ipc"]

resumen_deflactor = (
    base.dropna(subset=["deflactor_ipc"])
    .groupby("anio_periodo")["deflactor_ipc"].mean().round(4)
)
print("Deflactor medio por año (multiplicador a pesos constantes):")
print(resumen_deflactor)

Deflactor medio por año (multiplicador a pesos constantes):
anio_periodo
2020    1.4139
2021    1.3570
2022    1.2605
2023    1.1302
2024    1.0477
2025    0.9963
2026    0.9637
Name: deflactor_ipc, dtype: float64


## Capa 2 · Año y mes de gobierno

Los mandatos de alcalde en Colombia empiezan el **1 de enero**: Alfonso Eljach 2020–2023,
Jonathan Vásquez 2024–2027. Se calcula el **mes de gobierno** (1 a 48) de cada contrato para
poder comparar *posiciones equivalentes del mandato* en lugar de años calendario.

Se conserva además la marca de **cobertura confiable**: SECOP II fue adoptado gradualmente, por
lo que los primeros meses de Alfonso (2020 y parte de 2021) tienen registro casi nulo y **no**
deben leerse como "poca contratación", sino como "poca digitalización".


In [6]:
# 6. Mes y año de gobierno
INICIO_MANDATO = {
    "Alfonso Eljach": pd.Timestamp("2020-01-01"),
    "Jonathan Vasquez": pd.Timestamp("2024-01-01"),
}

def mes_de_gobierno(alcalde, fecha):
    if alcalde not in INICIO_MANDATO or pd.isna(fecha):
        return np.nan
    ini = INICIO_MANDATO[alcalde]
    return (fecha.year - ini.year) * 12 + (fecha.month - ini.month) + 1

base["mes_gobierno"] = [
    mes_de_gobierno(a, f) for a, f in zip(base["alcalde"], base["fecha_asignacion_periodo"])
]
base["mes_gobierno"] = base["mes_gobierno"].astype("Int64")
base["anio_gobierno"] = ((base["mes_gobierno"] - 1) // 12 + 1).astype("Int64")

# Primer mes de gobierno con cobertura confiable de la Alcaldía (abril 2021 = mes 16 de Alfonso)
MES_GOB_CONFIABLE_ALFONSO = 16   # abr-2021
print(base.groupby("alcalde")["anio_gobierno"].value_counts().sort_index())

alcalde           anio_gobierno
Alfonso Eljach    1                   7
                  2                3035
                  3                5855
                  4                4790
Jonathan Vasquez  1                4853
                  2                4991
                  3                4399
Name: count, dtype: Int64


## Capa 3 · Calendario electoral y ley de garantías

El análisis del ciclo político es un **componente principal** del proyecto. Se operacionalizan
aquí las variables que estaban descritas en los objetivos pero no existían en la base:

- **`ventana_ley_garantias`**: los ~4 meses previos a cada elección territorial, cuando la
  contratación directa (la mayoría de los CPS) queda restringida.
- **`tipo_anio_electoral`**: ordinario / preelectoral / electoral.
- **`dias_a_prox_eleccion`**: distancia al próximo comicio, para curvas de estacionalidad política.

Elecciones territoriales usadas como anclas: **27-oct-2019**, **29-oct-2023**, **29-oct-2027**.

> **Advertencia legal.** La marca de ventana se define por el calendario y sirve para *explicar
> estadísticamente* caídas y picos de contratación. No constituye por sí sola un juicio sobre el
> cumplimiento o incumplimiento de la norma en ningún contrato individual.


In [7]:
# 7. Calendario electoral y ventana de ley de garantías
ELECCIONES_TERRITORIALES = [
    pd.Timestamp("2019-10-27"),
    pd.Timestamp("2023-10-29"),
    pd.Timestamp("2027-10-29"),
]
MESES_RESTRICCION = 4  # ventana previa a la elección

def dias_a_proxima_eleccion(fecha):
    if pd.isna(fecha):
        return np.nan
    futuras = [(e - fecha).days for e in ELECCIONES_TERRITORIALES if e >= fecha]
    return min(futuras) if futuras else np.nan

def en_ventana_garantias(fecha):
    if pd.isna(fecha):
        return False
    for e in ELECCIONES_TERRITORIALES:
        if (e - pd.DateOffset(months=MESES_RESTRICCION)) <= fecha <= e:
            return True
    return False

base["dias_a_prox_eleccion"] = base["fecha_asignacion_periodo"].map(dias_a_proxima_eleccion)
base["ventana_ley_garantias"] = base["fecha_asignacion_periodo"].map(en_ventana_garantias)

ANIOS_ELECTORALES = {2019, 2023, 2027}
ANIOS_PREELECTORALES = {2018, 2022, 2026}
base["tipo_anio_electoral"] = np.select(
    [base["anio_periodo"].isin(ANIOS_ELECTORALES),
     base["anio_periodo"].isin(ANIOS_PREELECTORALES)],
    ["Electoral", "Preelectoral"],
    default="Ordinario",
)
print("Contratos por tipo de año electoral:")
print(base["tipo_anio_electoral"].value_counts())

Contratos por tipo de año electoral:
tipo_anio_electoral
Ordinario       17562
Preelectoral    13769
Electoral        6243
Name: count, dtype: int64


In [8]:
# 8. Reconstruir los universos analíticos con las nuevas capas
cps = base[base["es_cps_estricto"] == True].copy()
cps_alcaldia = cps[cps["es_alcaldia_analisis"] == True].copy()
print(f"CPS estricto: {len(cps):,}  |  CPS estricto Alcaldía: {len(cps_alcaldia):,}")

CPS estricto: 32,778  |  CPS estricto Alcaldía: 26,323


### Evidencia empírica de la ley de garantías (elección oct-2023)

La serie mensual de CPS de la Alcaldía muestra el mecanismo con nitidez: un **pico de
adjudicación en junio de 2023** (carrera previa al cierre) seguido de un **congelamiento casi
total entre julio y octubre de 2023** (ventana de restricción). Este patrón pertenece al *año 4
de Alfonso* y **no tiene equivalente** en el periodo de Jonathan (sin elección hasta 2027):
mezclarlos en una misma ventana distorsiona cualquier comparación de volumen.


In [9]:
# 9. Serie mensual de CPS Alcaldía alrededor de la elección de 2023
serie = (
    cps_alcaldia.assign(mes=lambda d: d["fecha_asignacion_periodo"].dt.to_period("M").astype(str))
    .groupby("mes")["contrato_llave"].nunique()
)
ventana_2023 = serie[(serie.index >= "2023-01") & (serie.index <= "2023-12")]
print("CPS Alcaldía por mes en 2023 (año electoral):")
print(ventana_2023)

reporte_garantias = (
    cps_alcaldia.groupby(["alcalde", "ventana_ley_garantias"])["contrato_llave"]
    .nunique().reset_index(name="cps")
)
display(reporte_garantias)

CPS Alcaldía por mes en 2023 (año electoral):
mes
2023-01     306
2023-02     509
2023-03     379
2023-04     235
2023-05     133
2023-06    2223
2023-08       1
2023-09       1
2023-10       4
2023-11     387
2023-12     252
Name: contrato_llave, dtype: int64


,alcalde,ventana_ley_garantias,cps
0,Alfonso Eljach,False,12805
1,Alfonso Eljach,True,5
2,Jonathan Vasquez,False,13513


## Capa 4 · Ventanas comparables alineadas por mandato

Se distinguen dos tipos de ventana:

| Ventana | Alfonso | Jonathan | ¿Alineada por gobierno? |
|---|---|---|---|
| 24 meses calendario (heredada) | 2022–2023 (meses 25–48) | 2024–2025 (meses 1–24) | **No** compara fin de mandato vs inicio |
| **Año 3 al mismo corte** (heredada) | ene–sep 2022 (meses 25–33) | ene–sep 2026 (meses 25–33) | **Sí** |
| **Meses de gobierno 16–33** (nueva) | abr-2021–sep-2022 | abr-2025–sep-2026 | **Sí**, y más larga (18 meses) |

La ventana calendario de 24 meses se **conserva** por continuidad, pero se marca como
secundaria. La comparación principal debe hacerse sobre **ventanas alineadas por mes de
gobierno**, que es el único terreno realmente equivalente dado que la cobertura confiable de
Alfonso empieza en su mes 16.


In [10]:
# 10. Definir la ventana alineada por mes de gobierno (intersección de cobertura confiable)
# Alfonso confiable desde mes 16; Jonathan disponible hasta mes 33 (corte sep-2026).
MES_GOB_INI_ALINEADA = 16
MES_GOB_FIN_ALINEADA = 33

base["ventana_alineada_gob_16_33"] = (
    base["mes_gobierno"].between(MES_GOB_INI_ALINEADA, MES_GOB_FIN_ALINEADA)
)
cps_alcaldia["ventana_alineada_gob_16_33"] = (
    cps_alcaldia["mes_gobierno"].between(MES_GOB_INI_ALINEADA, MES_GOB_FIN_ALINEADA)
)
N_MESES_ALINEADA = MES_GOB_FIN_ALINEADA - MES_GOB_INI_ALINEADA + 1
print(f"Ventana alineada: meses de gobierno {MES_GOB_INI_ALINEADA}-{MES_GOB_FIN_ALINEADA} "
      f"({N_MESES_ALINEADA} meses)")

Ventana alineada: meses de gobierno 16-33 (18 meses)


## Comparación nominal vs real 

La tabla siguiente  muestra el **mismo indicador** (valor mensual
mediano de los CPS) calculado en pesos nominales y en pesos constantes, en cada ventana.


In [11]:
# 11. Comparación nominal vs real del valor mensual, por ventana
def comparar_ventana(df, filtro, etiqueta):
    sub = df[filtro & df["tipo_cps_claro"]].copy()
    g = (
        sub.groupby("alcalde")
        .agg(cps=("contrato_llave", "nunique"),
             personas=("proveedor_llave", "nunique"),
             vm_nominal=("valor_mensual_equivalente", "median"),
             vm_real=("valor_mensual_real", "median"),
             dur_mediana=("duracion_meses_exacta", "median"))
        .reset_index()
    )
    g.insert(0, "ventana", etiqueta)
    return g

comparaciones = pd.concat([
    comparar_ventana(cps_alcaldia, cps_alcaldia["periodo_comparable_24m"],
                     "24m calendario (secundaria)"),
    comparar_ventana(cps_alcaldia, cps_alcaldia["periodo_anio3_mismo_corte"],
                     "Año 3 mismo corte (alineada)"),
    comparar_ventana(cps_alcaldia, cps_alcaldia["ventana_alineada_gob_16_33"],
                     "Meses gobierno 16-33 (alineada)"),
], ignore_index=True)

# Brecha porcentual Jonathan vs Alfonso, nominal y real
def brecha(tabla, col):
    piv = tabla.pivot(index="ventana", columns="alcalde", values=col)
    return ((piv["Jonathan Vasquez"] / piv["Alfonso Eljach"] - 1) * 100).round(1)

resumen_brechas = pd.DataFrame({
    "brecha_nominal_%": brecha(comparaciones, "vm_nominal"),
    "brecha_real_%": brecha(comparaciones, "vm_real"),
})
print("Valor mensual mediano por ventana (nominal vs real):")
display(comparaciones.round(0))
print("\nBrecha Jonathan vs Alfonso (+ = Jonathan mayor):")
display(resumen_brechas)

Valor mensual mediano por ventana (nominal vs real):


,ventana,alcalde,cps,personas,vm_nominal,vm_real,dur_mediana
0,24m calendario (secundaria),Alfonso Eljach,8987,4284,2705778.0,3282931.0,4.0
1,24m calendario (secundaria),Jonathan Vasquez,8581,4046,3493115.0,3492935.0,3.0
2,Año 3 mismo corte (alineada),Alfonso Eljach,3288,2638,2698091.0,3481227.0,4.0
3,Año 3 mismo corte (alineada),Jonathan Vasquez,4080,3073,3069580.0,2982502.0,4.0
4,Meses gobierno 16-33 (alineada),Alfonso Eljach,6907,3427,2638133.0,3481227.0,3.0
5,Meses gobierno 16-33 (alineada),Jonathan Vasquez,7629,4212,3044000.0,2982502.0,3.0



Brecha Jonathan vs Alfonso (+ = Jonathan mayor):


,brecha_nominal_%,brecha_real_%
ventana,,
24m calendario (secundaria),29.1,6.4
Año 3 mismo corte (alineada),13.8,-14.3
Meses gobierno 16-33 (alineada),15.4,-14.3


In [12]:
# 12. Recurrencia y personas por mes en la ventana alineada (en términos reales)
al = cps_alcaldia[cps_alcaldia["ventana_alineada_gob_16_33"]].copy()

personas_alc = (
    al.groupby(["alcalde", "proveedor_llave"])
    .agg(contratos=("contrato_llave", "nunique"),
         valor_real=("valor_contrato_real", "sum"))
    .reset_index()
)
resumen_alineada = (
    personas_alc.groupby("alcalde")
    .agg(personas=("proveedor_llave", "nunique"),
         contratos=("contratos", "sum"),
         contratos_por_persona=("contratos", "mean"),
         pct_2_o_mas=("contratos", lambda x: (x >= 2).mean() * 100))
    .reset_index()
)
resumen_alineada["cps_por_mes"] = (resumen_alineada["contratos"] / N_MESES_ALINEADA).round(1)
resumen_alineada["personas_por_mes"] = (resumen_alineada["personas"] / N_MESES_ALINEADA).round(1)
print("Resumen ventana alineada (meses de gobierno 16-33):")
display(resumen_alineada.round(2))

Resumen ventana alineada (meses de gobierno 16-33):


,alcalde,personas,contratos,contratos_por_persona,pct_2_o_mas,cps_por_mes,personas_por_mes
0,Alfonso Eljach,3677,7649,2.08,58.69,424.9,204.3
1,Jonathan Vasquez,4348,7961,1.83,50.92,442.3,241.6


## Guardado de resultados y manifiesto

Se guardan: la **base enriquecida** con las cuatro capas, las **tablas de comparación** listas
para gráficos, y un **manifiesto metodológico v5** que documenta las decisiones (fuente IPC, año
base, anclas electorales, ventanas). 


In [13]:
# 13. Guardar base enriquecida y tablas de comparación
RUTA_BASE_ENRIQUECIDA = RUTA_PROCESADOS / "03_base_analitica_enriquecida.parquet"
base.to_parquet(RUTA_BASE_ENRIQUECIDA, index=False)

RUTA_CPS_ENRIQUECIDO = RUTA_PROCESADOS / "03_cps_alcaldia_enriquecido.parquet"
cps_alcaldia.to_parquet(RUTA_CPS_ENRIQUECIDO, index=False)

comparaciones.to_csv(RUTA_TABLAS / "03_comparacion_valor_mensual_nominal_vs_real.csv",
                     index=False, encoding="utf-8-sig")
resumen_brechas.to_csv(RUTA_TABLAS / "03_brechas_nominal_vs_real.csv", encoding="utf-8-sig")
resumen_alineada.to_csv(RUTA_TABLAS / "03_resumen_ventana_alineada.csv",
                        index=False, encoding="utf-8-sig")
ventana_2023.to_csv(RUTA_TABLAS / "03_serie_cps_alcaldia_2023_electoral.csv",
                    encoding="utf-8-sig")
print("Guardado:", RUTA_BASE_ENRIQUECIDA.name, "+ 4 tablas en entregables/tablas")

Guardado: 03_base_analitica_enriquecida.parquet + 4 tablas en entregables/tablas


In [14]:
# 14. Manifiesto metodológico v5
manifiesto_v5 = {
    "version_base": "03_v5",
    "hereda_de": "02_v4",
    "corte_datos": "2026-09-06",
    "deflactor_ipc": {
        "fuente": "DANE - IPC base 2018, variacion anual diciembre-diciembre",
        "inflacion_dic": INFLACION_DIC,
        "inflacion_2026_proyectada": INFLACION_2026_PROYECTADA,
        "anio_base_pesos_constantes": ANIO_BASE,
        "metodo": "indice mensual por interpolacion geometrica entre diciembres",
        "nota": "Reemplazar por serie mensual oficial del IPC para la version de publicacion",
    },
    "alineacion_gobierno": {
        "inicio_mandato": {k: str(v.date()) for k, v in INICIO_MANDATO.items()},
        "mes_gobierno_confiable_alfonso": MES_GOB_CONFIABLE_ALFONSO,
    },
    "calendario_electoral": {
        "elecciones": [str(e.date()) for e in ELECCIONES_TERRITORIALES],
        "meses_restriccion_garantias": MESES_RESTRICCION,
    },
    "ventanas": {
        "24m_calendario": "SECUNDARIA - no alineada por mes de gobierno",
        "anio3_mismo_corte": "alineada (meses 25-33)",
        "gob_16_33": f"alineada, {N_MESES_ALINEADA} meses (meses de gobierno 16-33)",
        "recomendacion": "Usar ventanas alineadas y valores reales como comparacion principal",
    },
    "principio": "Los valores monetarios se comparan SOLO en pesos constantes. "
                 "Los volumenes se comparan SOLO en ventanas alineadas por mes de gobierno.",
}
with open(RUTA_INTERMEDIOS / "03_manifiesto_metodologico_v5.json", "w", encoding="utf-8") as f:
    json.dump(manifiesto_v5, f, ensure_ascii=False, indent=2, default=str)
print("Manifiesto v5 guardado.")

Manifiesto v5 guardado.
